# Data Ingestion Method 2: COPY INTO

## Overview
COPY INTO is a SQL statement that ingests data from files into an existing Delta table. It's idempotent, supporting incremental loads and better handling of duplicate data.

## Key Characteristics
- **Best for**: Incremental data loads, regular data ingestion
- **Idempotent**: Automatically skips duplicate files
- **Schema Management**: Table must exist beforehand
- **File Tracking**: Built-in duplicate detection
- **Flexibility**: Good for multiple file formats


## Prerequisites

1. Target Delta table must already exist
2. Schema must match source data
3. Access to source files/directory


## Step 1: Create Target Table Schema

First, we need to create an empty Delta table with the desired schema. We'll use transaction data that gets ingested in batches.

In [ ]:
-- Drop if exists for demo purposes
DROP TABLE IF EXISTS transactions_copy_into;

-- Create empty table with defined schema
CREATE TABLE transactions_copy_into (
  transaction_id STRING,
  customer_id STRING,
  amount DOUBLE,
  transaction_date DATE,
  payment_method STRING,
  status STRING,
  ingestion_timestamp TIMESTAMP
)
USING DELTA

In [ ]:
-- Verify empty table
SELECT COUNT(*) as row_count FROM transactions_copy_into

## Step 2: COPY INTO with Basic Options

Load data from CSV files into the Delta table.


In [ ]:
-- Load first batch of transactions
COPY INTO transactions_copy_into
FROM (
  SELECT 
    transaction_id,
    customer_id,
    amount,
    CAST(transaction_date AS DATE) as transaction_date,
    payment_method,
    status,
    CURRENT_TIMESTAMP() as ingestion_timestamp
  FROM read_files(
    '/Volumes/data_analysis/sample_data/transactions_batch1.csv',
    format => 'csv'
  )
)
FILEFORMAT = CSV

In [ ]:
-- Verify data loaded
SELECT COUNT(*) as total_rows FROM transactions_copy_into

In [ ]:
-- View sample data
SELECT * FROM transactions_copy_into LIMIT 5

## Step 3: Idempotency - Running COPY INTO with New Batch

COPY INTO automatically detects and skips files that have already been loaded. Now we load the second batch.

In [ ]:
-- Load second batch of transactions
COPY INTO transactions_copy_into
FROM (
  SELECT 
    transaction_id,
    customer_id,
    amount,
    CAST(transaction_date AS DATE) as transaction_date,
    payment_method,
    status,
    CURRENT_TIMESTAMP() as ingestion_timestamp
  FROM read_files(
    '/Volumes/data_analysis/sample_data/transactions_batch2.csv',
    format => 'csv'
  )
)
FILEFORMAT = CSV

In [ ]:
-- Row count should increase (new data added from batch 2)
SELECT COUNT(*) as total_rows FROM transactions_copy_into

## Step 4: COPY INTO with Advanced Options


In [ ]:
-- Drop and recreate for demo
DROP TABLE IF EXISTS transactions_copy_advanced;

CREATE TABLE transactions_copy_advanced (
  transaction_id STRING,
  customer_id STRING,
  amount DOUBLE,
  transaction_date DATE,
  payment_method STRING,
  status STRING,
  ingestion_timestamp TIMESTAMP,
  _ingestion_source STRING
)
USING DELTA

In [ ]:
-- COPY INTO with additional metadata and source tracking
COPY INTO transactions_copy_advanced
FROM (
  SELECT 
    transaction_id,
    customer_id,
    amount,
    CAST(transaction_date AS DATE) as transaction_date,
    payment_method,
    status,
    CURRENT_TIMESTAMP() as ingestion_timestamp,
    'BATCH_LOAD' as _ingestion_source
  FROM read_files(
    '/Volumes/data_analysis/sample_data/transactions_batch1.csv',
    format => 'csv'
  )
)
FILEFORMAT = CSV

In [ ]:
SELECT * FROM transactions_copy_advanced LIMIT 5

## Advantages of COPY INTO

✅ **Idempotent**: Automatically handles duplicate files

✅ **Incremental Loads**: Can load new files without re-ingesting old data

✅ **Flexible**: Works with various file formats (CSV, Parquet, JSON)

✅ **Schema Control**: Define schema explicitly before loading

✅ **Data Validation**: Can include validation logic in the SELECT statement

✅ **Batch Operations**: Good for batch file processing


## Disadvantages of COPY INTO

❌ **Manual Table Creation**: Schema must be defined upfront

❌ **Latency**: Not ideal for real-time streaming

❌ **File Format Requirements**: Files must match expected format

❌ **No Auto-Detection**: Cannot handle schema changes automatically


## Use Cases for COPY INTO

1. **Regular Batch Loads**: Daily/weekly file uploads from external systems
2. **Incremental Ingestion**: Appending new files to existing tables
3. **Data Migration**: Migrating data from data lakes or S3
4. **Multi-format Loading**: Loading data from various file types
5. **ETL Workflows**: Part of scheduled data pipelines


## Summary

COPY INTO is ideal for:
- Incremental data loads
- Idempotent batch operations
- Regular scheduled data ingestion

For streaming data and automatic schema evolution, consider **Auto Loader** instead.
